# Glitch setelah potongan pertama — uji urutan pemanggilan

Mode 30 detik juga dilaporkan glitch setelah potongan pertama. Memperpanjang chunk
belum menyelesaikan masalah. Jangan ulang seluruh lagu untuk uji ini.

Dengarkan **sumber, stem tersimpan, dan raw model** dari chunk 1 dan chunk 0.
Lalu jalankan dua inferensi dengan **urutan dibalik: [1,0]**, dalam satu proses baru.
Input disalin byte-for-byte tanpa gain/resampling tambahan; seed, Base, 50 langkah,
CFG 7, MPS dan CPU offload sama dengan run sebelumnya. Tidak ada perubahan metadata.

Untuk lagu normal: B = 27–57 detik, A = 0–30 detik. Posisi sebenarnya dicetak
sesuai cache. Model baru memproses B terlebih dahulu, kemudian A. Tidak ada stitching
pada hasil uji, tidak ada unduhan model, dan hasil/cache lama dipertahankan.
Dua inferensi berjalan saat Run All; set RUN_REPLAY=False untuk hanya mendengarkan
hasil cache. Buka dari checkout repository, dengan runtime notebook 02 yang sudah siap.


In [ ]:
from pathlib import Path
import json, sys, uuid
from IPython.display import Audio, display
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "tools/stem_eval"))
from sequence_diagnostics import prepare_sequence
from mac_workflow import check_model_runtime
from setup_runtime import runtime_python
from stream_process import run_logged
from seam_diagnostics import signal_stats
from audio import read_pcm
RUN = None  # atau Path folder full_song30_* yang bermasalah
RUN_REPLAY = True
if RUN is None:
    candidates = sorted((ROOT / "runs").glob("full_song30_*/experiment.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    candidates = [p for p in candidates if json.loads(p.read_text()).get("status") == "generated"]
    assert candidates, "Pilih RUN dari notebook 06 yang memiliki hasil chunk 0 dan 1"
    RUN = candidates[0].parent
WORK = ROOT / "runs" / ("sequence_diagnostics_" + uuid.uuid4().hex[:12])
comparison = prepare_sequence(RUN, WORK / "cached")
print("Run lama:", RUN)
print("Folder diagnostik:", WORK)
for record in comparison["records"]:
    print("Chunk asli", record["original_chunk_index"], "mulai", record["start_seconds"], "detik; urutan baru", record["sequence_position"] + 1)


## 1. Dengarkan cache sebelum mengulang model

Pastikan sumber B terdengar normal. Dengarkan stem B sendiri: glitch yang sudah
ada pada file ini perlu ditelusuri sebelum stitching. Jika stem/raw bersih tetapi
lagu gabungan rusak, fokuskan pemeriksaan pada stitching/mix. Raw mungkin memiliki
sample rate berbeda; file disalin utuh tanpa konversi.


In [ ]:
for record in comparison["records"]:
    index = record["original_chunk_index"]
    preview = WORK / "cached" / f"cached_{index:03d}"
    for label, path in [("SUMBER", preview / "source.wav"), ("STEM TERSIMPAN", preview / "stem.wav"), *[("RAW MODEL TERSIMPAN", p) for p in sorted(preview.glob("raw_*.wav"))]]:
        print(label, "chunk", index, "mulai", record["start_seconds"], "detik")
        display(Audio(filename=str(path)))
    print("Statistik sumber:", record["source_stats"])
    print("Statistik stem lama:", record["cached_stem_stats"])


## 2. Ulang dua input dengan urutan B → A

Dua pemanggilan memakai satu handler baru. Ini mengubah urutan pemrosesan sambil
menjaga input dan parameter. Kedua hasil diputar terpisah, tanpa crossfade.
Tidak mengklaim model bebas glitch; hasil MPS dapat bervariasi meski seed sama.


In [ ]:
REPLAY = WORK / "reversed"
if RUN_REPLAY:
    print(json.dumps(check_model_runtime(runtime_python()), indent=2))
    command = [str(runtime_python()), str(ROOT / "scripts/sequence_diagnostics.py"),
               "--run", str(RUN), "--out", str(REPLAY), "--execute"]
    run_logged(command, check=True)
else:
    print("CACHE ONLY: tidak ada inferensi atau unduhan model")


## 3. Bandingkan hasil, lalu tentukan langkah berikutnya

- **B bersih saat pertama; A rusak saat kedua:** mengarah ke pengaruh urutan/state
  runtime. Perlu konfirmasi ulang sebelum menerapkan proses terpisah per chunk.
- **B rusak saat pertama; A bersih saat kedua:** mengarah ke isi/konteks B atau
  generasi numeriknya; proses terpisah saja belum terbukti membantu.
- **Keduanya bersih:** pengaruh sesi lama atau variasi inferensi mungkin berperan.
- **Keduanya rusak:** periksa sumber, parameter, dtype/decoder dan raw model.

Statistik peak/RMS bukan penilaian musik. Laporkan hasil B saat urutan pertama,
A saat urutan kedua, dan apakah raw model sudah glitch. Metadata MPS/versi/setting
tersimpan pada `reversed/session.json`, `comparison.json`, dan result tiap job.
Ini uji diagnosis, belum perbaikan otomatis atau generasi ulang lagu penuh.


In [ ]:
summary = []
if RUN_REPLAY:
    report = json.loads((REPLAY / "comparison.json").read_text())
    for position, job in enumerate(report["jobs"]):
        folder = REPLAY / f"{position:03d}_A_{job['instrument']}_seed{job['seed']}"
        result = json.loads((folder / "result.json").read_text())
        assert result["status"] == "generated"
        assert result["source_sha256"] == comparison["jobs"][position]["source_sha256"]
        stem = folder / result["stem_file"]
        print("BARU: urutan", position + 1, "chunk asli", job["original_chunk_index"], "mulai", job["start_frame"] / 32000, "detik")
        stats = signal_stats(read_pcm(stem))
        summary.append({"sequence_position": position, "original_chunk_index": job["original_chunk_index"], "source_sha256": result["source_sha256"], "stem_stats": stats})
        print(json.dumps(stats, indent=2))
        display(Audio(filename=str(stem)))
        display(Audio(filename=str(stem.parent / "mix.wav")))
        for raw in sorted(stem.parent.glob("*.wav")):
            if raw.name not in {"stem.wav", "mix.wav"}:
                print("RAW MODEL BARU:", raw.name)
                display(Audio(filename=str(raw)))
    (WORK / "listening_summary.json").write_text(json.dumps(summary, indent=2))
else:
    print("Hanya hasil lama diputar; uji urutan belum dijalankan")
